# Chapter 4 · Time Series with Linear Models (Notebook)

Turn a recorded signal into a table, fit a simple autoregressive model, and test honestly whether it can forecast.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Rathachai/DA-LAB/blob/gh-pages/learnings/linear/notebooks/nb-04-time-series.ipynb)

- Lecture: [en-04-time-series.md](https://github.com/Rathachai/DA-LAB/blob/gh-pages/learnings/linear/en-04-time-series.md)
- Lab: [LM401](https://rathachai.github.io/DA-LAB/learnings/linear/lm401.html)

Powered by: Sonnet  |  AI Whisperer: rathachai.github.io

## 0. Setup
pandas, numpy and scikit-learn do the work; matplotlib only draws the plots. Run the cells from top to bottom.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error, r2_score
plt.rcParams["figure.figsize"] = (9, 4)
print("Libraries ready")

## 1. Load the data
`lm401_timeseries.csv` holds ten 200-point signals (the same ones as in the LM401 lab). Column `t` is the time index, the other columns are signals, like channels from a data logger.

In [ ]:
url = "https://rathachai.github.io/DA-LAB/learnings/linear/data/lm401_timeseries.csv"
df = pd.read_csv(url)
print(df.shape)
df.head()

A quick summary of each signal: its typical level (mean), its spread (std), its smallest and largest values.

In [ ]:
df.describe().round(2)

## 2. Look at all ten signals
Before modelling, always look. Notice the trend, the repeating season, the random walk that wanders, and the white noise that has no pattern.

In [ ]:
names = [c for c in df.columns if c != "t"]
fig, axes = plt.subplots(5, 2, figsize=(11, 12), sharex=True)
for ax, name in zip(axes.ravel(), names):
    ax.plot(df["t"], df[name], lw=1.2)
    ax.set_title(name)
    ax.set_ylabel("value")
for ax in axes[-1]:
    ax.set_xlabel("time index t")
fig.suptitle("The ten LM401 series", y=1.0)
plt.tight_layout(); plt.show()

## 3. A series is ONE array
A series is a single list `y0, y1, ..., yn` in time order. There are no input columns yet and no target column. Here are the first 8 values of `trend_season`, as a one-column table and as a one-row table.

In [ ]:
y = df["trend_season"].to_numpy()
print("array length n+1 =", len(y))
print("one column:")
display(df[["t", "trend_season"]].head(8))
print("one row:")
display(pd.DataFrame([y[:8]], columns=[f"y{i}" for i in range(8)]))

## 4. The sliding window
`make_window(series, w)` builds the table. For each target index `t >= w`: the inputs are the previous `w` values (`x1` oldest ... `xw` newest) and the answer `y` is the value that comes next.

In [ ]:
def make_window(series, w):
    s = pd.Series(np.asarray(series, dtype=float))
    lags = {f"x{k}": s.shift(w - k + 1) for k in range(1, w + 1)}   # x1 = oldest (shift w), xw = newest (shift 1)
    out = pd.concat(lags, axis=1)
    out["y"] = s                                                    # target = the next value
    out = out.dropna()                                              # first w rows have no full window
    out.insert(0, "t", out.index)                                   # t = index of the target
    return out.reset_index(drop=True)

small = [10, 12, 13, 15, 18, 20]       # the lecture's numeric example
make_window(small, 3)

Six values with `w = 3` give `6 - 3 = 3` rows. Neighbouring rows repeat the same numbers, shifted by one place. Now a real series: `trend_season` with `w = 5`.

In [ ]:
w = 5
win = make_window(df["trend_season"], w)
print("shape:", win.shape, "= (n+1-w, w+1) with n+1 =", len(df), "and w =", w, "-> (", len(df) - w, ",", w + 1, ")  (plus the t column)")
win.head()

The course also ships the same table ready-made. Do our rows match it?

In [ ]:
url_w = "https://rathachai.github.io/DA-LAB/learnings/linear/data/lm401_window_w5_trend_season.csv"
ready = pd.read_csv(url_w)
cols = ["t"] + [f"x{k}" for k in range(1, 6)] + ["y"]
print("ready-made shape:", ready.shape)
print("same values:", np.allclose(win[cols].to_numpy(), ready[cols].to_numpy()))
ready.head()

## 5. Fit an AR(w) model
An autoregressive model predicts the next value from the last `w` values: `y_hat = b0 + b1*x1 + ... + bw*xw`. It is a difference equation whose coefficients are estimated from data. Here we fit on all rows only to read the coefficients; we will fit properly after splitting.

In [ ]:
feats = [f"x{k}" for k in range(1, w + 1)]
ar = LinearRegression().fit(win[feats], win["y"])
coef = pd.DataFrame({"term": ["intercept"] + feats,
                     "coefficient": [ar.intercept_] + list(ar.coef_)}).round(4)
display(coef)
eq = " + ".join(f"{c:.3f}*x{k}" for k, c in enumerate(ar.coef_, 1))
print(f"y_hat = {ar.intercept_:.3f} + {eq}")
print("x5 is the newest sample, x1 the oldest.")

## 6. Chronological split (70 / 30)
Real forecasting predicts the future from the past, so we train on the early rows and test on the later rows. Never shuffle.

In [ ]:
n_train = int(round(0.7 * len(win)))
train, test = win.iloc[:n_train], win.iloc[n_train:]
print("train rows:", len(train), " test rows:", len(test))
print("matches the ready-made split column:", (ready["split"].iloc[:n_train] == "train").all() and (ready["split"].iloc[n_train:] == "test").all())
split_t = test["t"].iloc[0]
plt.plot(train["t"], train["y"], label="train (early)")
plt.plot(test["t"], test["y"], lw=2.5, label="test (late)")
plt.axvline(split_t, color="gray", ls="--")
plt.title("Chronological split of trend_season"); plt.xlabel("time index t"); plt.ylabel("value"); plt.legend(); plt.show()

## 7. One-step-ahead predictions (using actual data)
Fit on the training rows only. At every time the model is given the real last 5 values and predicts the next one, so errors do not pile up. We score with MAE, RMSE, MAPE (%) and R².

In [ ]:
def metrics(y_true, y_pred):
    return {"MAE": mean_absolute_error(y_true, y_pred),
            "RMSE": mean_squared_error(y_true, y_pred) ** 0.5,
            "MAPE %": 100 * mean_absolute_percentage_error(y_true, y_pred),
            "R2": r2_score(y_true, y_pred)}

model = LinearRegression().fit(train[feats], train["y"])
p_train = model.predict(train[feats])
p_test = model.predict(test[feats])
res = pd.DataFrame({"train": metrics(train["y"], p_train), "test": metrics(test["y"], p_test)}).T
display(res.round(3))
plt.plot(train["t"], train["y"], color="gray", lw=1, label="actual")
plt.plot(test["t"], test["y"], color="gray", lw=2.5)
plt.plot(train["t"], p_train, color="tab:red", lw=1, label="prediction (train, thin)")
plt.plot(test["t"], p_test, color="tab:red", lw=2.5, label="prediction (test, thick)")
plt.title("One-step-ahead AR(5) on trend_season"); plt.xlabel("time index t"); plt.ylabel("value"); plt.legend(); plt.show()

## 8. Naive baseline: persistence
The simplest rule is "next value = last value" (`x5`). A model is only useful if it beats this. We compare on `trend_season` and on `random_walk`.

In [ ]:
def evaluate(name, w=5, frac=0.7):
    tab = make_window(df[name], w)
    f = [f"x{k}" for k in range(1, w + 1)]
    k = int(round(frac * len(tab)))
    tr, te = tab.iloc[:k], tab.iloc[k:]
    m = LinearRegression().fit(tr[f], tr["y"])
    return {"series": name,
            "model MAE": mean_absolute_error(te["y"], m.predict(te[f])),
            "persistence MAE": mean_absolute_error(te["y"], te[f[-1]])}

cmp_ = pd.DataFrame([evaluate("trend_season"), evaluate("random_walk")]).set_index("series")
cmp_["model beats baseline?"] = cmp_["model MAE"] < cmp_["persistence MAE"]
cmp_.round(3)

For a random walk the best guess is the last value, so the model cannot do much better than persistence. Small differences there are not real skill.

## 9. Recursive multi-step forecast
Now pretend we have **no new measurements** during the test period. We start from the last 5 known values, predict one step, feed the prediction back in, and repeat (dead reckoning). We also keep the "last known value" baseline.

In [ ]:
window = list(test[feats].iloc[0])          # last 5 real values before the test period
rec = []
for _ in range(len(test)):
    nxt = model.predict(pd.DataFrame([window], columns=feats))[0]
    rec.append(nxt)
    window = window[1:] + [nxt]               # slide: drop oldest, add prediction
rec = np.array(rec)
last_known = np.full(len(test), test[feats].iloc[0, -1])
out = pd.DataFrame({"one-step": metrics(test["y"], p_test),
                    "recursive": metrics(test["y"], rec),
                    "last-known baseline": metrics(test["y"], last_known)}).T
display(out.round(3))
plt.plot(train["t"], train["y"], color="lightgray", label="train")
plt.plot(test["t"], test["y"], color="black", lw=2, label="actual")
plt.plot(test["t"], p_test, color="tab:red", label="one-step (actual data)")
plt.plot(test["t"], rec, color="tab:blue", ls="--", lw=2, label="recursive forecast")
plt.title("Actual vs one-step vs recursive"); plt.xlabel("time index t"); plt.ylabel("value"); plt.legend(); plt.show()

Error versus horizon: the one-step error stays small, while the recursive error jumps as soon as we stop feeding real data. (On a seasonal series it can then wander up and down instead of growing steadily; on a trend it keeps growing.)

In [ ]:
h = np.arange(1, len(test) + 1)
err_rec = np.abs(test["y"].to_numpy() - rec)
err_one = np.abs(test["y"].to_numpy() - p_test)
plt.plot(h, err_one, label="one-step |error|")
plt.plot(h, err_rec, label="recursive |error|")
plt.title("Absolute error vs forecast horizon"); plt.xlabel("steps ahead (horizon)"); plt.ylabel("absolute error"); plt.legend(); plt.show()
print("one-step  mean error, first 10 steps:", err_one[:10].mean().round(2))
print("recursive mean error, first 10 steps:", err_rec[:10].mean().round(2))
print("recursive mean error, last 10 steps :", err_rec[-10:].mean().round(2))

## 10. Window-size experiment
Is a bigger window always better? We try `w = 1..10` on `sine` and `trend_season` and plot the test MAE (one-step, chronological split).

In [ ]:
rows = []
for name in ["sine", "trend_season"]:
    for ww in range(1, 11):
        tab = make_window(df[name], ww)
        f = [f"x{k}" for k in range(1, ww + 1)]
        k = int(round(0.7 * len(tab)))
        m = LinearRegression().fit(tab[f].iloc[:k], tab["y"].iloc[:k])
        rows.append({"series": name, "w": ww,
                     "test MAE": mean_absolute_error(tab["y"].iloc[k:], m.predict(tab[f].iloc[k:]))})
wexp = pd.DataFrame(rows)
piv = wexp.pivot(index="w", columns="series", values="test MAE")
display(piv.round(3))
for name in piv.columns:
    plt.plot(piv.index, piv[name], marker="o", label=name)
plt.title("Test MAE vs window size w"); plt.xlabel("window size w"); plt.ylabel("test MAE"); plt.legend(); plt.show()

A sine wave needs at least 2 lags (position and direction). After that, extra lags add little.

## 11. Random split vs chronological split (leakage)
Neighbouring windows share most of their values. If we shuffle rows, a test row is almost a copy of a training row, so the test error looks better than it really is. We compare both splits (random split averaged over 20 shuffles).

In [ ]:
def split_errors(name, w=5):
    tab = make_window(df[name], w)
    f = [f"x{k}" for k in range(1, w + 1)]
    k = int(round(0.7 * len(tab)))
    m = LinearRegression().fit(tab[f].iloc[:k], tab["y"].iloc[:k])
    chrono = mean_absolute_error(tab["y"].iloc[k:], m.predict(tab[f].iloc[k:]))
    rnd = []
    for seed in range(20):
        Xa, Xb, ya, yb = train_test_split(tab[f], tab["y"], test_size=0.3, random_state=seed)
        rnd.append(mean_absolute_error(yb, LinearRegression().fit(Xa, ya).predict(Xb)))
    return {"series": name, "chronological MAE": chrono, "random MAE": np.mean(rnd)}

leak = pd.DataFrame([split_errors(s) for s in ["trend_season", "ar1", "random_walk", "sine"]]).set_index("series")
leak["random looks better by %"] = 100 * (1 - leak["random MAE"] / leak["chronological MAE"])
display(leak.round(3))
leak[["chronological MAE", "random MAE"]].plot.bar(rot=0)
plt.title("Test MAE: chronological vs random split"); plt.xlabel("series"); plt.ylabel("test MAE"); plt.show()

A random split reports an error that differs from the honest chronological one, and often looks better. The linear model is simple, so the leakage effect can be modest; flexible models suffer far more. The rule stays the same: for forecasting, split chronologically.

## 12. Summary
- A time series is **one array** of values in time order; level, trend, seasonality and noise make it up.
- The **sliding window** turns it into a table of shape `(n+1-w, w+1)`; a linear model on lags is an **AR(w)** model.
- **Split chronologically**; random splits leak the future.
- **One-step** predictions use real data; **recursive** forecasts reuse predictions, so error grows with the horizon.
- Always compare with the **naive baseline**; for a random walk it is hard to beat.
- Bigger `w` is not always better: a sine needs only 2 lags.

Powered by: Sonnet  |  AI Whisperer: rathachai.github.io